# Validation automatique des mesures morphométriques

Pour chaque mesure (27), on classe l'image en **mesurable / non mesurable** à partir des
confiances de keypoints prédites par le modèle de pose.

**Convention : la classe positive est `non mesurable`** (classe minoritaire, celle qui nous intéresse).

**8 approches comparées :**

| clé | features |
|---|---|
| `seuil_conf_moy` | moyenne des confiances des kp directs de la mesure |
| `seuil_conf_min` | minimum des confiances des kp directs de la mesure |
| `xgb_direct` / `rf_direct` | confiances des kp directs + one-hot groupe |
| `xgb_related` / `rf_related` | confiances du voisinage anatomique (`related_entities`) + one-hot groupe |
| `xgb_all` / `rf_all` | confiances des 42 kp + one-hot groupe |

**Protocole :** 5-fold stratifié, prédictions out-of-fold. Le seuil de décision est choisi
sur une partition interne du *train* (maximisation du MCC), jamais sur le test.

**Métrique de comparaison inter-mesures : le MCC** (robuste au déséquilibre), complété par
l'AP normalisée `(AP - prévalence) / (1 - prévalence)` qui corrige l'effet de prévalence.

## 1. Configuration

In [19]:
from __future__ import annotations

import gc
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
)
from xgboost import XGBClassifier
import joblib

from insect_anatomy import INSECT_GROUPS, MEAS_TO_KP, MEASUREMENTS, POINTS, related_entities
from dataset import STATUS_SUFFIX, build_dataset, label_report

warnings.filterwarnings("ignore")
matplotlib.use("Agg")  # pas de fenêtre : on écrit des PNG

# --- chemins (à adapter) ---------------------------------------------------
DATA_DIR = Path("./data")                       # CSV d'annotation
DATABASE_DIR = Path("../../databases/full databases")               # arborescence images/<groupe>/
RESULTS_PATH = Path("../process_folder/results_10.csv")  # sorties du modèle de pose
OUT_DIR = Path("outputs")

# --- protocole -------------------------------------------------------------
N_FOLDS = 5
RANDOM_STATE = 0
MIN_MINORITY = 20   # garde-fou : mesure ignorée en dessous de ce nombre
NA_FILL = -1.0      # sentinelle d'imputation pour la random forest

for sub in ("pr_curves", "confusion"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)


def slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

## 2. Chargement des données

In [20]:
frame, columns = build_dataset(DATA_DIR, DATABASE_DIR, RESULTS_PATH)
GROUP_COLS = [f"{g}_one_hot" for g in INSECT_GROUPS]

print(f"{len(frame)} images | {columns.summary()}")
frame[GROUP_COLS].sum().rename("n images").to_frame().T

Dropping 2 annotation rows whose image is absent from the database


                                       image_name  total length_status  \
0  IMG_0093_specimen_1_MECKON_NEON.BET.D20.000001                    1   
1  IMG_0093_specimen_3_MECKON_NEON.BET.D20.000004                    1   
2  IMG_0095_specimen_2_MECKON_NEON.BET.D20.000007                    1   
3  IMG_0095_specimen_3_MECKON_NEON.BET.D20.000010                    1   
4  IMG_0109_specimen_1_MECKON_NEON.BET.D20.000011                    1   

   head width_status  head length_status  inter ocular distance_status  \
0                  1                   1                             1   
1                  1                   1                             1   
2                  1                   1                             1   
3                  1                   1                             1   
4                  1                   1                             1   

   right antenna length_status  left antenna length_status  \
0                            1                  

,coleoptera_one_hot,diptera_one_hot,hymenoptera_one_hot,lepidoptera_one_hot
n images,397,287,241,274


In [22]:
# Prévalence par mesure : sert au garde-fou et à la lecture des accuracies.
prevalence = []
for measure in MEASUREMENTS:
    status = f"{measure}{STATUS_SUFFIX}"
    if status not in frame.columns:
        continue
    y = 1 - frame[status].astype(int).to_numpy()   # 1 = non mesurable
    prevalence.append({
        "measure": measure,
        "n": len(y),
        "n_unmeasurable": int(y.sum()),
        "prevalence_unmeasurable": float(y.mean()),
        "kept": bool(min(y.sum(), len(y) - y.sum()) >= MIN_MINORITY),
    })

prevalence = pd.DataFrame(prevalence)
prevalence.to_csv(OUT_DIR / "prevalence.csv", index=False)

KEPT = prevalence.loc[prevalence["kept"], "measure"].tolist()
SKIPPED = prevalence.loc[~prevalence["kept"], "measure"].tolist()
print(f"{len(KEPT)} mesures retenues, {len(SKIPPED)} ignorées (< {MIN_MINORITY} exemples minoritaires)")
print("ignorées :", SKIPPED)
prevalence

26 mesures retenues, 1 ignorées (< 20 exemples minoritaires)
ignorées : ['thorax length']


,measure,n,n_unmeasurable,prevalence_unmeasurable,kept
0,total length,1199,313,0.261051,True
1,head width,1199,281,0.234362,True
2,head length,1199,174,0.145121,True
3,inter ocular distance,1199,321,0.267723,True
4,right antenna length,1199,180,0.150125,True
5,left antenna length,1199,150,0.125104,True
6,thorax width,1199,202,0.168474,True
7,thorax length,1199,5,0.004170,False
8,abdomen width,1199,261,0.217681,True
9,abdomen length,1199,150,0.125104,True


## 3. Jeux de features et approches

In [23]:
def conf_columns(points) -> list:
    """Colonnes de confiance existantes pour une liste de keypoints."""
    return [columns.conf[p] for p in points if p in columns.conf]


ALL_CONF = conf_columns(POINTS)


def feature_sets(measure: str) -> dict:
    return {
        "direct": conf_columns(MEAS_TO_KP[measure]),
        "related": conf_columns(related_entities(measure)[0]),
        "all": ALL_CONF,
    }


def target(measure: str) -> np.ndarray:
    """1 = non mesurable (classe positive, minoritaire)."""
    return 1 - frame[f"{measure}{STATUS_SUFFIX}"].astype(int).to_numpy()


def make_xgb(y_train: np.ndarray) -> XGBClassifier:
    n_pos = max(int(y_train.sum()), 1)
    n_neg = max(len(y_train) - n_pos, 1)
    return XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=n_neg / n_pos,          # déséquilibre
        eval_metric="logloss", tree_method="hist",
        n_jobs=-1, random_state=RANDOM_STATE,
    )


def make_rf(y_train: np.ndarray) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2,
        class_weight="balanced",                  # déséquilibre
        n_jobs=-1, random_state=RANDOM_STATE,
    )


# (clé, type, jeu de features, fabrique de modèle)
APPROACHES = [
    #("seuil_conf_moy", "rule", "direct", None),
    #("seuil_conf_min", "rule", "direct", None),
    #("xgb_direct", "model", "direct", make_xgb),
    #("rf_direct", "model", "direct", make_rf),
    #("xgb_related", "model", "related", make_xgb),
    ("rf_related", "model", "related", make_rf),
    #("xgb_all", "model", "all", make_xgb),
    #("rf_all", "model", "all", make_rf),
]
NAMES = [a[0] for a in APPROACHES]

## 4. Score, seuil et validation croisée

Toutes les approches produisent un **score croissant avec la probabilité de « non mesurable »**,
ce qui rend courbes PR et matrices de confusion directement comparables.
Un keypoint non détecté (cellule vide) vaut une confiance nulle pour les règles de seuil,
reste `NaN` pour XGBoost (branche informative native) et est imputé à `-1` pour la random forest.

In [24]:
def rule_score(values: pd.DataFrame, how: str) -> np.ndarray:
    data = np.nan_to_num(values.to_numpy(dtype=float), nan=0.0)  # kp manquant -> conf 0
    agg = data.mean(axis=1) if how == "mean" else data.min(axis=1)
    return -agg   # confiance basse -> score élevé -> non mesurable


def best_threshold(y: np.ndarray, score: np.ndarray) -> float:
    """Seuil maximisant le MCC, cherché sur des données non vues à l'entraînement."""
    candidates = np.unique(np.quantile(score, np.linspace(0.0, 1.0, 201)))
    best_t, best_m = candidates[0], -np.inf
    for t in candidates:
        m = matthews_corrcoef(y, (score >= t).astype(int))
        if m > best_m:
            best_t, best_m = t, m
    return float(best_t)


def evaluate_measure(measure: str) -> dict:
    """Prédictions out-of-fold des 8 approches pour une mesure."""
    y = target(measure)
    sets = feature_sets(measure)
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    scores = {name: np.zeros(len(y)) for name in NAMES}
    preds = {name: np.zeros(len(y), dtype=int) for name in NAMES}
    thresholds = {name: [] for name in NAMES}

    for train_idx, test_idx in splitter.split(np.zeros(len(y)), y):
        # partition interne : sert uniquement à fixer le seuil de décision
        inner_fit, inner_val = train_test_split(
            train_idx, test_size=0.25, stratify=y[train_idx], random_state=RANDOM_STATE
        )
        for name, kind, which, factory in APPROACHES:
            cols = sets[which]
            if not cols:
                continue
            if kind == "rule":
                how = "mean" if name.endswith("moy") else "min"
                val_score = rule_score(frame.iloc[inner_val][cols], how)
                test_score = rule_score(frame.iloc[test_idx][cols], how)
            else:
                data = frame[cols + GROUP_COLS]
                if name.startswith("rf"):
                    data = data.fillna(NA_FILL)   # la RF ne gère pas les NaN
                model = factory(y[inner_fit])
                model.fit(data.iloc[inner_fit], y[inner_fit])
                if name == "rf_related":
                    joblib.dump(model, f"models/rf_related_{measure}.joblib")
                val_score = model.predict_proba(data.iloc[inner_val])[:, 1]
                del model
                gc.collect()
                model = factory(y[train_idx])
                model.fit(data.iloc[train_idx], y[train_idx])
                test_score = model.predict_proba(data.iloc[test_idx])[:, 1]
                del model
                gc.collect()

            threshold = best_threshold(y[inner_val], val_score)
            scores[name][test_idx] = test_score
            preds[name][test_idx] = (test_score >= threshold).astype(int)
            thresholds[name].append(threshold)

    return {"y": y, "scores": scores, "preds": preds, "thresholds": thresholds}

## 5. Métriques et figures

In [25]:
def metric_rows(measure: str, result: dict) -> list:
    y = result["y"]
    prev = float(y.mean())
    rows = []
    for name in NAMES:
        score, pred = result["scores"][name], result["preds"][name]
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        ap = float(average_precision_score(y, score))
        rows.append({
            "measure": measure,
            "model": name,
            "n": len(y),
            "prevalence_unmeasurable": prev,
            "mcc": float(matthews_corrcoef(y, pred)),
            "average_precision": ap,
            "average_precision_norm": (ap - prev) / (1 - prev) if prev < 1 else np.nan,
            "accuracy": float((tp + tn) / len(y)),
            "accuracy_unmeasurable": float(tp / (tp + fn)) if (tp + fn) else np.nan,
            "accuracy_measurable": float(tn / (tn + fp)) if (tn + fp) else np.nan,
            "balanced_accuracy": float(0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))),
            "precision_unmeasurable": float(tp / (tp + fp)) if (tp + fp) else np.nan,
            "threshold_median": float(np.median(result["thresholds"][name])),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })
    return rows


def plot_pr(measure: str, result: dict) -> None:
    y = result["y"]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for name in NAMES:
        score = result["scores"][name]
        precision, recall, _ = precision_recall_curve(y, score)
        ap = average_precision_score(y, score)
        ax.plot(recall, precision, lw=1.6, label=f"{name} (AP={ap:.3f})")
    ax.axhline(y.mean(), color="grey", ls="--", lw=1, label=f"hasard ({y.mean():.3f})")
    ax.set_xlabel("Rappel (non mesurable)")
    ax.set_ylabel("Précision (non mesurable)")
    ax.set_title(f"Courbe précision-rappel — {measure}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "pr_curves" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)


def plot_confusion(measure: str, result: dict) -> None:
    y = result["y"]
    labels = ["mesurable", "non mes."]
    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax, name in zip(axes.ravel(), NAMES):
        matrix = confusion_matrix(y, result["preds"][name], labels=[0, 1])
        normalised = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)
        ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{matrix[i, j]}\n{normalised[i, j]:.0%}",
                        ha="center", va="center", fontsize=9,
                        color="white" if normalised[i, j] > 0.5 else "black")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_title(f"{name}\nMCC={matthews_corrcoef(y, result['preds'][name]):.3f}", fontsize=9)
        ax.set_xlabel("prédit", fontsize=8)
        ax.set_ylabel("réel", fontsize=8)
    fig.suptitle(f"Matrices de confusion (out-of-fold) — {measure}")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "confusion" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)

## 6. Boucle principale

In [26]:
rows, OOF = [], {}                        # <- au lieu de rows = []
for i, measure in enumerate(KEPT, start=1):
    print(f"[{i:2d}/{len(KEPT)}] {measure}", flush=True)
    result = evaluate_measure(measure)
    rows.extend(metric_rows(measure, result))
    plot_pr(measure, result)
    plot_confusion(measure, result)
    OOF[measure] = {"y": result["y"], **{n: result["preds"][n] for n in NAMES}}   # <- ajout
    del result
    gc.collect()

metrics = pd.DataFrame(rows)
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
print(f"\n{len(metrics)} lignes écrites dans {OUT_DIR / 'metrics.csv'}")
metrics.head(8)

[ 1/26] total length
[ 2/26] head width
[ 3/26] head length
[ 4/26] inter ocular distance
[ 5/26] right antenna length
[ 6/26] left antenna length
[ 7/26] thorax width
[ 8/26] abdomen width
[ 9/26] abdomen length
[10/26] intertegular distance
[11/26] left hind wing length
[12/26] right hind wing length
[13/26] left hind wing width
[14/26] right hind wing width
[15/26] left fore wing length
[16/26] right fore wing length
[17/26] left fore wing width
[18/26] right fore wing width
[19/26] left hind leg length
[20/26] left hind leg femur length
[21/26] left hind leg tibia length
[22/26] left hind leg tarsus length
[23/26] right hind leg length
[24/26] right hind leg femur length
[25/26] right hind leg tibia length
[26/26] right hind leg tarsus length

26 lignes écrites dans outputs/metrics.csv


,measure,model,n,prevalence_unmeasurable,mcc,average_precision,average_precision_norm,accuracy,accuracy_unmeasurable,accuracy_measurable,balanced_accuracy,precision_unmeasurable,threshold_median,tn,fp,fn,tp
0,total length,rf_related,1199,0.261051,0.466091,0.659719,0.539507,0.779817,0.674121,0.817156,0.745639,0.565684,0.400573,724,162,102,211
1,head width,rf_related,1199,0.234362,0.684537,0.823955,0.770068,0.893244,0.622776,0.976035,0.799405,0.888325,0.591014,896,22,106,175
2,head length,rf_related,1199,0.145121,0.601682,0.712963,0.664236,0.904087,0.632184,0.950244,0.791214,0.683230,0.520371,974,51,64,110
3,inter ocular distance,rf_related,1199,0.267723,0.679102,0.826897,0.763610,0.879900,0.626168,0.972665,0.799417,0.893333,0.574273,854,24,120,201
4,right antenna length,rf_related,1199,0.150125,0.492946,0.572254,0.496696,0.882402,0.483333,0.952895,0.718114,0.644444,0.555327,971,48,93,87
5,left antenna length,rf_related,1199,0.125104,0.509043,0.574969,0.514192,0.899917,0.513333,0.955195,0.734264,0.620968,0.518515,1002,47,73,77
6,thorax width,rf_related,1199,0.168474,0.744689,0.858767,0.830152,0.932444,0.702970,0.978937,0.840954,0.871166,0.619030,976,21,60,142
7,abdomen width,rf_related,1199,0.217681,0.660457,0.835432,0.789641,0.885738,0.720307,0.931770,0.826038,0.746032,0.495426,874,64,73,188


## 7. Comparaison des approches

Classement par **rang moyen du MCC** sur l'ensemble des mesures retenues
(rang 1 = meilleure approche pour la mesure), plus le nombre de victoires.

In [27]:
mcc = metrics.pivot(index="measure", columns="model", values="mcc")[NAMES]
ranks = mcc.rank(axis=1, ascending=False)

ranking = pd.DataFrame({
    "mean_rank": ranks.mean(),
    "median_mcc": mcc.median(),
    "mean_mcc": mcc.mean(),
    "mean_ap_norm": metrics.pivot(index="measure", columns="model",
                                  values="average_precision_norm")[NAMES].mean(),
    "mean_accuracy_unmeasurable": metrics.pivot(index="measure", columns="model",
                                                values="accuracy_unmeasurable")[NAMES].mean(),
    "wins": mcc.idxmax(axis=1).value_counts().reindex(NAMES).fillna(0).astype(int),
}).sort_values("mean_rank")

ranking.to_csv(OUT_DIR / "ranking.csv")
ranking

,mean_rank,median_mcc,mean_mcc,mean_ap_norm,mean_accuracy_unmeasurable,wins
rf_related,1.0,0.670158,0.641114,0.76026,0.827347,26


In [9]:
# Heatmap mesure x approche (MCC)
fig, ax = plt.subplots(figsize=(9, 0.42 * len(mcc) + 2.5))
image = ax.imshow(mcc.to_numpy(), cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(NAMES)))
ax.set_yticks(range(len(mcc)))
ax.set_xticklabels(NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(mcc.index, fontsize=8)
for i in range(mcc.shape[0]):
    for j in range(mcc.shape[1]):
        value = mcc.iat[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if value < 0.6 else "black")
fig.colorbar(image, ax=ax, label="MCC")
ax.set_title("MCC out-of-fold par mesure et par approche")
fig.tight_layout()
fig.savefig(OUT_DIR / "heatmap_mcc.png", dpi=140)
plt.close(fig)

# Barplot du rang moyen
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(ranking.index[::-1], ranking["mean_rank"][::-1], color="steelblue")
for name, value in zip(ranking.index[::-1], ranking["mean_rank"][::-1]):
    ax.text(value + 0.05, name, f"{value:.2f}", va="center", fontsize=8)
ax.set_xlabel(f"MCC mean rank on {len(mcc)} measures (1 = best)")
ax.set_title("Approach ranking")
fig.tight_layout()
fig.savefig(OUT_DIR / "ranking.png", dpi=140)
plt.close(fig)

print("Figures écrites :", len(list(OUT_DIR.rglob('*.png'))))

Figures écrites : 83


In [10]:
# --- Tableau récapitulatif : MCC, précision, rappel (classe "non mesurable") ---
summary = (
    metrics
    .rename(columns={
        "precision_unmeasurable": "precision",
        "accuracy_unmeasurable": "recall",
    })
    .groupby("model")[["mcc", "precision", "recall"]]
    .agg(["mean", "std"])
    .reindex(NAMES)
    .sort_values(("mcc", "mean"), ascending=False)
    .round(3)
)
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary.insert(0, "n_mesures", metrics.groupby("model")["measure"].nunique().reindex(summary.index))

summary.to_csv(OUT_DIR / "summary_mcc_precision_recall.csv")
summary

,n_mesures,mcc_mean,mcc_std,precision_mean,precision_std,recall_mean,recall_std
model,,,,,,,
rf_all,26,0.646,0.201,0.822,0.186,0.839,0.159
rf_related,26,0.641,0.178,0.811,0.186,0.827,0.161
xgb_all,26,0.631,0.204,0.806,0.198,0.830,0.167
xgb_related,26,0.630,0.182,0.802,0.197,0.826,0.158
rf_direct,26,0.582,0.183,0.736,0.223,0.826,0.164
xgb_direct,26,0.580,0.182,0.717,0.242,0.848,0.151
seuil_conf_min,26,0.425,0.290,0.672,0.302,0.636,0.323
seuil_conf_moy,26,0.405,0.300,0.670,0.299,0.642,0.307


In [11]:
# --- Conséquence pratique : mesures exploitables par image -----------------
BASELINE = "seuil_conf_min"          # "avant" : seuil sur la confiance minimale
BEST = ranking.index[1]              # "après" : rasoir d'okham : privilegier le model le plus simple a perf egale
print(f"avant = {BASELINE} | après = {BEST}")

y_true = np.column_stack([OOF[m]["y"] for m in KEPT])        # 1 = non mesurable
valid = y_true == 0                                          # mesure réellement exploitable

rows, per_image = [], {"image_name": frame["image_name"], "group": frame["group"].astype(str)}


def summarise(label, rejected):
    kept = ~rejected
    kept_valid = kept & valid
    per_image[f"n_conservees_{label}"] = kept.sum(axis=1)
    per_image[f"n_valides_conservees_{label}"] = kept_valid.sum(axis=1)
    rows.append({
        "filtrage": label,
        "mesures_conservees_par_image": kept.sum(axis=1).mean(),
        "dont_reellement_valides": kept_valid.sum(axis=1).mean(),
        "valides_perdues_par_image": (valid & rejected).sum(axis=1).mean(),
        "retention_des_valides": kept_valid.sum() / valid.sum(),
        "contamination_des_conservees": (kept & ~valid).sum() / max(kept.sum(), 1),
    })


summarise("sans_filtrage", np.zeros_like(y_true, dtype=bool))
for name in (BASELINE, BEST):
    predicted = np.column_stack([OOF[m][name] for m in KEPT])
    summarise(name, predicted == 1)

practical = pd.DataFrame(rows).set_index("filtrage").round(3)
practical.to_csv(OUT_DIR / "impact_filtrage.csv")
pd.DataFrame(per_image).to_csv(OUT_DIR / "mesures_par_image.csv", index=False)

print(f"\n{len(KEPT)} mesures évaluées, {valid.mean():.1%} réellement exploitables\n")
practical

avant = seuil_conf_min | après = rf_related

26 mesures évaluées, 51.8% réellement exploitables



,mesures_conservees_par_image,dont_reellement_valides,valides_perdues_par_image,retention_des_valides,contamination_des_conservees
filtrage,,,,,
sans_filtrage,26.000,13.458,0.000,1.000,0.482
seuil_conf_min,13.765,11.273,2.185,0.838,0.181
rf_related,13.053,11.912,1.545,0.885,0.087


In [18]:
# --- Nombre de variables d'entrée par approche -----------------------------
counts = []
for measure in KEPT:
    sets = feature_sets(measure)
    for name, kind, which, _ in APPROACHES:
        counts.append({
            "measure": measure,
            "model": name,
            "n_features": len(sets[which]) + (len(GROUP_COLS) if kind == "model" else 0),
        })

n_features = (
    pd.DataFrame(counts)
    .groupby("model")["n_features"]
    .agg(mean_n_features="mean", min_n_features="min", max_n_features="max")
    .reindex(NAMES)
    .round(2)
)
n_features["mean_mcc"] = metrics.groupby("model")["mcc"].mean().round(3)
n_features = n_features.sort_values("mean_n_features")

n_features.to_csv(OUT_DIR / "n_features.csv")
n_features


,mean_n_features,min_n_features,max_n_features,mean_mcc
model,,,,
seuil_conf_moy,2.31,2,4,0.405
seuil_conf_min,2.31,2,4,0.425
xgb_direct,6.31,6,8,0.580
rf_direct,6.31,6,8,0.582
xgb_related,12.62,8,24,0.630
rf_related,12.62,8,24,0.641
xgb_all,46.00,46,46,0.631
rf_all,46.00,46,46,0.646


## 8. Figures complémentaires

In [12]:
# --- Répartition des labels par mesure (barres empilées) -------------------
order = prevalence.sort_values("prevalence_unmeasurable")
n_ok = (order["n"] - order["n_unmeasurable"]).to_numpy()
n_ko = order["n_unmeasurable"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 0.34 * len(order) + 2))
ax.barh(order["measure"], n_ok, color="#4c9f70", label="measurable")
ax.barh(order["measure"], n_ko, left=n_ok, color="#c0504d", label="non measurable")
for i, (a, b) in enumerate(zip(n_ok, n_ko)):
    ax.text(a + b + order["n"].max() * 0.01, i, f"{b / (a + b):.0%}", va="center", fontsize=7)
ax.set_xlim(0, order["n"].max() * 1.09)   # marge pour les étiquettes de pourcentage
ax.set_xlabel("number of images")
ax.set_title("Label repartition per measure (sorted by non-measurable ratio)")
ax.tick_params(labelsize=8)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT_DIR / "labels_par_mesure.png", dpi=140)
plt.close(fig)
print("écrit :", OUT_DIR / "labels_par_mesure.png")

écrit : outputs/labels_par_mesure.png


In [14]:
# --- MCC par mesure : xgb_all vs rf_all ------------------------------------
PAIR = ["xgb_related", "rf_related"]
pair_mcc = mcc[PAIR].sort_values("xgb_related")
positions = np.arange(len(pair_mcc))
height = 0.38

fig, ax = plt.subplots(figsize=(9, 0.42 * len(pair_mcc) + 2))
ax.barh(positions + height / 2, pair_mcc["xgb_related"], height=height,
        color="#4472c4", label="xgb_related")
ax.barh(positions - height / 2, pair_mcc["rf_related"], height=height,
        color="#ed7d31", label="rf_related")
ax.set_yticks(positions)
ax.set_yticklabels(pair_mcc.index, fontsize=8)
ax.set_xlim(min(0.0, float(pair_mcc.min().min()) - 0.05), 1.0)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("MCC (out-of-fold)")
ax.set_title("MCC per measure : models trained on all keypoints and measures")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT_DIR / "mcc_xgb_related_vs_rf_related.png", dpi=140)
plt.close(fig)
print("écrit :", OUT_DIR / "mcc_xgb_related_vs_rf_related.png")

écrit : outputs/mcc_xgb_related_vs_rf_related.png


### Importance des variables

Le modèle le mieux classé (hors règles de seuil, qui n'ont pas de variables) est réentraîné
sur l'intégralité des données, mesure par mesure. Ces importances sont **descriptives** :
elles servent à voir *quels keypoints portent le signal*, pas à mesurer une performance.
Deux réserves classiques : les importances par impureté (random forest) favorisent les
variables à forte cardinalité, et deux confiances corrélées se partagent le crédit.

In [15]:
MODEL_APPROACHES = {a[0]: (a[2], a[3]) for a in APPROACHES if a[1] == "model"}
BEST_MODEL = next(name for name in ranking.index if name in MODEL_APPROACHES)
BEST_MODEL = BEST
WHICH, FACTORY = MODEL_APPROACHES[BEST_MODEL]
print(f"Meilleur modèle : {BEST_MODEL} (features : {WHICH})")

(OUT_DIR / "importance").mkdir(exist_ok=True)
CONF_TO_POINT = {column: point for point, column in columns.conf.items()}


def pretty(column: str) -> str:
    return CONF_TO_POINT.get(column, column.replace("_one_hot", ""))


series = []
for measure in KEPT:
    cols = feature_sets(measure)[WHICH]
    if not cols:
        continue
    y = target(measure)
    data = frame[cols + GROUP_COLS]
    if BEST_MODEL.startswith("rf"):
        data = data.fillna(NA_FILL)

    model = FACTORY(y)
    model.fit(data, y)
    values = pd.Series(model.feature_importances_, index=[pretty(c) for c in data.columns])
    del model
    gc.collect()
    series.append(values.rename(measure))

    top = values.sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(6.5, 0.32 * len(top) + 1.6))
    ax.barh(top.index, top.to_numpy(), color="steelblue")
    ax.set_xlabel("importance")
    ax.set_title(f"{BEST_MODEL} : {measure}", fontsize=10)
    ax.tick_params(labelsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "importance" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)

importance = pd.concat(series, axis=1).T
importance.to_csv(OUT_DIR / "feature_importance.csv")
print(f"{len(importance)} mesures, {importance.shape[1]} variables")

Meilleur modèle : rf_related (features : related)
26 mesures, 46 variables


In [16]:
# Importance moyenne sur l'ensemble des mesures
mean_importance = importance.mean().sort_values().tail(20)
coverage = importance.notna().sum()

fig, ax = plt.subplots(figsize=(7, 0.32 * len(mean_importance) + 2))
ax.barh(mean_importance.index, mean_importance.to_numpy(), color="#4472c4")
for i, name in enumerate(mean_importance.index):
    ax.text(mean_importance[name], i, f"  {coverage[name]}/{len(importance)}",
            va="center", fontsize=7, color="grey")
ax.set_xlim(0, float(mean_importance.max()) * 1.12)
ax.set_xlabel("mean importance (grey number : number of measure where variable exists)")
ax.set_title(f"Most used variables ({BEST_MODEL})")
ax.tick_params(labelsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "importance_moyenne.png", dpi=140)
plt.close(fig)
print("écrit :", OUT_DIR / "importance_moyenne.png")

écrit : outputs/importance_moyenne.png


## 9. Lecture des résultats

- `outputs/metrics.csv` — toutes les métriques, une ligne par (mesure, approche).
- `outputs/prevalence.csv` — effectifs et mesures écartées par le garde-fou.
- `outputs/ranking.csv` + `ranking.png` — classement global des 8 approches.
- `outputs/heatmap_mcc.png` — quelles mesures sont difficiles, indépendamment de l'approche.
- `outputs/pr_curves/*.png`, `outputs/confusion/*.png` — détail par mesure.
- `outputs/labels_par_mesure.png` — répartition mesurable / non mesurable.
- `outputs/mcc_xgb_all_vs_rf_all.png` — comparaison directe des deux modèles complets.
- `outputs/importance/*.png`, `outputs/importance_moyenne.png`, `feature_importance.csv`.

Deux points de vigilance à l'interprétation :

1. `accuracy` doit toujours être lue face à `1 - prevalence_unmeasurable` (accuracy du
   classifieur constant). Une accuracy de 0,95 sur une mesure à 5 % de non-mesurables ne vaut rien.
2. Les mesures d'ailes postérieures sont labellisées non mesurables *par construction* chez les
   groupes qui n'en ont pas : le MCC y sera très élevé simplement parce que le one-hot de groupe
   suffit. Le tableau ci-dessous isole ces cas.

In [17]:
# Mesures dont le label est quasi déterminé par le groupe taxonomique.
suspects = []
for measure in KEPT:
    y = pd.Series(target(measure))
    by_group = y.groupby(frame["group"].astype(str).to_numpy()).mean()
    if ((by_group < 0.05) | (by_group > 0.95)).all():
        suspects.append({"measure": measure, **by_group.round(2).to_dict()})

suspects = pd.DataFrame(suspects)
if not suspects.empty:
    suspects.to_csv(OUT_DIR / "group_determined_measures.csv", index=False)
    print("Taux de non-mesurable par groupe (mesures triviales) :")
suspects

""


In [ ]:
# save model

import joblib

joblib.dump(rf, "my_random_forest.joblib")